# VQGAN Utilities

Optional taming-transformers functionality without import-time downloads or global gradient changes.

## Imports and defaults

Import shared numerical, model, image, and typing tools before defining reusable concepts.

In [ ]:
#| export
"""VQGAN asset loading and image-token conversion utilities."""

import sys
from pathlib import Path
from typing import Any, Protocol

import requests
import torch
from omegaconf import OmegaConf
from PIL import Image
from torch import Tensor, nn
from torchvision.transforms import functional as transform_functional

VQGAN_CHECKPOINT_URL: str = (
    "https://heibox.uni-heidelberg.de/f/140747ba53464f49b476/?dl=1"
)
VQGAN_CONFIG_URL: str = (
    "https://heibox.uni-heidelberg.de/f/6ecf2af6c658432c8298/?dl=1"
)
DEFAULT_CHECKPOINT_PATH: Path = Path("files/vqgan_imagenet_f16_1024.ckpt")
DEFAULT_CONFIG_PATH: Path = Path("files/vqgan_imagenet_f16_1024.yaml")
DEFAULT_DEVICE: torch.device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
device: str = DEFAULT_DEVICE.type


## VQGAN interface and asset download

Describe required model operations and download assets explicitly.

In [ ]:
#| export
class VQGANLike(Protocol):
    """Structural interface used by the image/token conversion helpers."""

    encoder: nn.Module
    quant_conv: nn.Module
    post_quant_conv: nn.Module
    decoder: nn.Module
    quantize: Any


def _download_file(url: str, destination: Path, timeout: float) -> None:
    """Download one VQGAN asset when it is absent."""
    if destination.exists():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    destination.write_bytes(response.content)


def download_vqgan_assets(
    checkpoint_path: str | Path = DEFAULT_CHECKPOINT_PATH,
    config_path: str | Path = DEFAULT_CONFIG_PATH,
    timeout: float = 60.0,
) -> tuple[Path, Path]:
    """Download the pretrained VQGAN checkpoint and configuration if needed.

    Args:
        checkpoint_path: Local checkpoint destination.
        config_path: Local YAML configuration destination.
        timeout: Per-request HTTP timeout in seconds.

    Returns:
        Resolved checkpoint and configuration paths.

    Raises:
        requests.HTTPError: If either download returns an error response.
    """
    checkpoint = Path(checkpoint_path)
    config = Path(config_path)
    _download_file(VQGAN_CHECKPOINT_URL, checkpoint, timeout)
    _download_file(VQGAN_CONFIG_URL, config, timeout)
    return checkpoint, config


## Optional model loading

Import taming-transformers lazily and restore its checkpoint.

In [ ]:
#| export
def load_model(
    checkpoint_path: str | Path = DEFAULT_CHECKPOINT_PATH,
    config_path: str | Path = DEFAULT_CONFIG_PATH,
    device: torch.device | str = DEFAULT_DEVICE,
    taming_transformers_path: str | Path | None = None,
) -> nn.Module:
    """Load the pretrained ImageNet VQGAN model.

    Assets are downloaded lazily on the first call, avoiding network and global
    gradient side effects during module import.

    Args:
        checkpoint_path: VQGAN checkpoint path.
        config_path: VQGAN YAML configuration path.
        device: Destination device.
        taming_transformers_path: Optional checkout of CompVis
            ``taming-transformers`` to add to ``sys.path``.

    Returns:
        Loaded VQGAN in evaluation mode.

    Raises:
        ImportError: If ``taming-transformers`` is unavailable.
    """
    checkpoint, config_file = download_vqgan_assets(
        checkpoint_path, config_path
    )
    if taming_transformers_path is not None:
        dependency_path = str(Path(taming_transformers_path))
        if dependency_path not in sys.path:
            sys.path.append(dependency_path)
    try:
        from taming.models.vqgan import VQModel
    except ImportError as error:
        raise ImportError(
            "load_model requires CompVis taming-transformers; clone it and pass "
            "taming_transformers_path"
        ) from error

    config: Any = OmegaConf.load(config_file)
    model = VQModel(**config.model.params).to(device)
    checkpoint_data: dict[str, Any] = torch.load(
        checkpoint, map_location=device
    )
    model.load_state_dict(checkpoint_data["state_dict"], strict=False)
    return model.eval()


## Image preprocessing

Resize and center-crop images into normalized VQGAN inputs.

In [ ]:
#| export
def process_image(file_name: str | Path, size: int = 384) -> Tensor:
    """Load an image and prepare a normalized square VQGAN input.

    Args:
        file_name: Source image path.
        size: Output height and width.

    Returns:
        Image tensor shaped ``(1, channels, size, size)`` in ``[-1, 1]``.
    """
    with Image.open(file_name) as source_image:
        shortest_side: int = min(source_image.size)
        scale: float = size / shortest_side
        resized_shape: tuple[int, int] = (
            round(scale * source_image.size[1]),
            round(scale * source_image.size[0]),
        )
        resized = transform_functional.resize(
            source_image,
            resized_shape,
            interpolation=Image.Resampling.LANCZOS,
        )
        cropped = transform_functional.center_crop(resized, [size, size])
        image_tensor: Tensor = transform_functional.to_tensor(cropped)
    return image_tensor.unsqueeze(0) * 2 - 1


## Image-token conversion

Quantize images to IDs and decode IDs through the codebook.

In [ ]:
#| export
@torch.no_grad()
def image_to_sequence(
    model: VQGANLike,
    image: Tensor,
    device: torch.device | str = DEFAULT_DEVICE,
) -> Tensor:
    """Quantize an image into a sequence of codebook indices.

    Args:
        model: Loaded VQGAN model.
        image: Input shaped ``(batch_size, channels, height, width)``.
        device: Model input device.

    Returns:
        Integer codebook indices produced by the VQGAN quantizer.
    """
    encoded: Tensor = model.encoder(image.to(device))
    projected: Tensor = model.quant_conv(encoded)
    _, _, indices = model.quantize(projected)
    return indices[2]


@torch.no_grad()
def sequence_to_image(model: VQGANLike, sequence: Tensor) -> Tensor:
    """Decode VQGAN codebook indices into an image tensor.

    Args:
        model: Loaded VQGAN model.
        sequence: Codebook indices representing a 24-by-24 latent grid.

    Returns:
        Reconstructed image tensor.
    """
    quantized: Tensor = model.quantize.embedding(sequence)
    # (num_tokens, channels) -> (1, channels, latent_height, latent_width)
    quantized = quantized.permute(1, 0).view((1, 256, 24, 24))
    post_quantized: Tensor = model.post_quant_conv(quantized)
    return model.decoder(post_quantized)


## Summary

The utilities make optional assets explicit while retaining preprocessing, quantization, and decoding.